# Preliminaries: Linear Algebra Foundations for Numerical Linear Algebra

This notebook provides a rigorous, computational, and visual introduction to the linear algebra prerequisites for a numerical linear algebra (NLA) course. Every concept here reappears—often in disguise—throughout QR factorization, eigenvalue algorithms, iterative solvers, and the SVD.

**Topics covered:**
1. Vector Spaces and the Four Fundamental Subspaces
2. Linear Independence, Span, Basis, and Dimension
3. Linear Maps and Matrix Representations
4. Inner Products and Norms
5. Orthogonal Projections
6. Eigenvalues, Eigenvectors, and the Spectral Theorem
7. Rank and Near-Rank Deficiency

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.linalg import null_space, orth, svd

np.set_printoptions(precision=6, suppress=True)
plt.rcParams.update({'figure.figsize': (8, 5), 'font.size': 12})

---
## 1. Vector Spaces and the Four Fundamental Subspaces

A **vector space** $V$ over $\mathbb{R}$ is a set equipped with addition and scalar multiplication satisfying the usual axioms (closure, associativity, commutativity, identity, inverses, distributivity). A **subspace** $W \subseteq V$ is a non-empty subset that is itself a vector space under the inherited operations — equivalently, $W$ is closed under addition and scalar multiplication.

### The Four Fundamental Subspaces

Given $A \in \mathbb{R}^{m \times n}$ with $\text{rank}(A) = r$, the four fundamental subspaces are:

| Subspace | Definition | Dimension | Lives in |
|---|---|---|---|
| **Column space** $\mathcal{C}(A)$ | $\{A\mathbf{x} : \mathbf{x} \in \mathbb{R}^n\}$ | $r$ | $\mathbb{R}^m$ |
| **Null space** $\mathcal{N}(A)$ | $\{\mathbf{x} \in \mathbb{R}^n : A\mathbf{x} = \mathbf{0}\}$ | $n - r$ | $\mathbb{R}^n$ |
| **Row space** $\mathcal{C}(A^T)$ | $\{A^T\mathbf{y} : \mathbf{y} \in \mathbb{R}^m\}$ | $r$ | $\mathbb{R}^n$ |
| **Left null space** $\mathcal{N}(A^T)$ | $\{\mathbf{y} \in \mathbb{R}^m : A^T\mathbf{y} = \mathbf{0}\}$ | $m - r$ | $\mathbb{R}^m$ |

### Key relationships
- $\mathcal{C}(A) \perp \mathcal{N}(A^T)$ and $\mathcal{C}(A) \oplus \mathcal{N}(A^T) = \mathbb{R}^m$
- $\mathcal{C}(A^T) \perp \mathcal{N}(A)$ and $\mathcal{C}(A^T) \oplus \mathcal{N}(A) = \mathbb{R}^n$
- **Rank–Nullity Theorem:** $\text{rank}(A) + \text{nullity}(A) = n$

The matrix $A$ is a **map** from $\mathbb{R}^n$ to $\mathbb{R}^m$. Vectors in the row space get mapped bijectively onto the column space; vectors in the null space get annihilated.

In [ ]:
# --- Four Fundamental Subspaces ---
# A rank-2 matrix in R^{3x4}: m=3, n=4, r=2
A = np.array([
    [1, 2, 0, 1],
    [0, 1, 1, 0],
    [1, 3, 1, 1]   # row 3 = row 1 + row 2 => rank = 2
], dtype=float)

m, n = A.shape
r = np.linalg.matrix_rank(A)

print(f"A is {m}x{n}, rank = {r}")
print(f"Expected dimensions: C(A)={r}, N(A)={n-r}, C(A^T)={r}, N(A^T)={m-r}")
print()

# Column space basis (via orth, which uses SVD internally)
col_space = orth(A)
print(f"Column space basis (dim {col_space.shape[1]}):")
print(col_space)
print()

# Null space basis
null_sp = null_space(A)
print(f"Null space basis (dim {null_sp.shape[1]}):")
print(null_sp)
print()

# Row space = column space of A^T
row_space = orth(A.T)
print(f"Row space basis (dim {row_space.shape[1]}):")
print(row_space)
print()

# Left null space = null space of A^T
left_null = null_space(A.T)
print(f"Left null space basis (dim {left_null.shape[1]}):")
print(left_null)
print()

# Verify rank-nullity theorem
print(f"Rank-Nullity check: rank(A) + nullity(A) = {r} + {null_sp.shape[1]} = {r + null_sp.shape[1]} = n = {n}  ✓")

In [ ]:
# Verify orthogonality relationships between fundamental subspaces
# C(A) ⊥ N(A^T)  and  C(A^T) ⊥ N(A)

print("Orthogonality verification:")
print(f"  C(A)^T @ N(A^T) (should be ≈ 0):\n{col_space.T @ left_null}\n")
print(f"  C(A^T)^T @ N(A) (should be ≈ 0):\n{row_space.T @ null_sp}\n")

# Verify that null space vectors are indeed annihilated by A
print("Null space verification (A @ null_vec should be 0):")
for i in range(null_sp.shape[1]):
    print(f"  A @ n_{i+1} = {A @ null_sp[:, i]}")

In [ ]:
# Visualize the four fundamental subspaces for a 2x2 rank-1 matrix
# This lets us draw everything in 2D
A2 = np.array([[2, 1],
               [4, 2]], dtype=float)  # rank 1

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Domain space R^2: row space and null space
ax = axes[0]
ax.set_title(r"Domain $\mathbb{R}^2$: Row space & Null space", fontsize=13)
ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)

# Row space of A2 = span of rows = span([2,1])
t = np.linspace(-2.5, 2.5, 100)
row_dir = np.array([2, 1]) / np.linalg.norm([2, 1])
ax.plot(t * row_dir[0], t * row_dir[1], 'b-', lw=2, label=r'$\mathcal{C}(A^T)$ (row space)')
ax.annotate('', xy=1.8*row_dir, xytext=(0,0),
            arrowprops=dict(arrowstyle='->', color='blue', lw=2))

# Null space of A2 = span([-1, 2]) (perpendicular to row direction)
null_dir = np.array([-1, 2]) / np.linalg.norm([-1, 2])
ax.plot(t * null_dir[0], t * null_dir[1], 'r--', lw=2, label=r'$\mathcal{N}(A)$ (null space)')
ax.annotate('', xy=1.8*null_dir, xytext=(0,0),
            arrowprops=dict(arrowstyle='->', color='red', lw=2))
ax.legend(fontsize=10)

# Codomain space R^2: column space and left null space
ax = axes[1]
ax.set_title(r"Codomain $\mathbb{R}^2$: Column space & Left null space", fontsize=13)
ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)

# Column space = span([2,4]) = span([1,2])
col_dir = np.array([1, 2]) / np.linalg.norm([1, 2])
ax.plot(t * col_dir[0], t * col_dir[1], 'g-', lw=2, label=r'$\mathcal{C}(A)$ (col space)')
ax.annotate('', xy=1.8*col_dir, xytext=(0,0),
            arrowprops=dict(arrowstyle='->', color='green', lw=2))

# Left null space = span([-2, 1]) (perpendicular to column direction)
lnull_dir = np.array([-2, 1]) / np.linalg.norm([-2, 1])
ax.plot(t * lnull_dir[0], t * lnull_dir[1], 'm--', lw=2, label=r'$\mathcal{N}(A^T)$ (left null)')
ax.annotate('', xy=1.8*lnull_dir, xytext=(0,0),
            arrowprops=dict(arrowstyle='->', color='purple', lw=2))
ax.legend(fontsize=10)

plt.suptitle("Four Fundamental Subspaces of a Rank-1 Matrix", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 2. Linear Independence, Span, Basis, and Dimension

### Definitions

- **Span:** $\text{span}\{v_1, \dots, v_k\} = \{\alpha_1 v_1 + \cdots + \alpha_k v_k : \alpha_i \in \mathbb{R}\}$. The set of all linear combinations.
- **Linear independence:** Vectors $\{v_1, \dots, v_k\}$ are linearly independent iff $\alpha_1 v_1 + \cdots + \alpha_k v_k = 0 \implies \alpha_1 = \cdots = \alpha_k = 0$. Equivalently, no vector in the set can be written as a linear combination of the others.
- **Basis:** A linearly independent set that spans $V$. Every vector in $V$ has a **unique** representation as a linear combination of basis vectors.
- **Dimension:** $\dim(V)$ = the number of vectors in any basis of $V$. All bases have the same size (a theorem, not a definition).

### Why this matters for NLA
Everything in NLA is ultimately about **choosing good bases**:
- QR factorization finds an orthonormal basis for $\mathcal{C}(A)$
- The SVD finds orthonormal bases for *both* the row space and column space simultaneously
- Krylov methods build a basis for a subspace iteratively
- Conditioning depends on how "nearly dependent" basis vectors are

In [ ]:
# --- Linear Independence ---
# Test whether a set of vectors is linearly independent by checking if the
# matrix they form has full column rank.

v1 = np.array([1, 0, 2])
v2 = np.array([0, 1, 1])
v3 = np.array([1, 1, 3])   # v3 = v1 + v2 => dependent

V_dep = np.column_stack([v1, v2, v3])
V_ind = np.column_stack([v1, v2, np.array([0, 0, 1])])

print("Dependent set {v1, v2, v3 = v1+v2}:")
print(f"  Matrix:\n{V_dep}")
print(f"  Rank = {np.linalg.matrix_rank(V_dep)} (< 3 columns => dependent)")
print(f"  Singular values: {np.linalg.svd(V_dep, compute_uv=False)}")
print(f"  Note: smallest singular value ≈ 0 signals dependence\n")

print("Independent set {v1, v2, [0,0,1]}:")
print(f"  Matrix:\n{V_ind}")
print(f"  Rank = {np.linalg.matrix_rank(V_ind)} (= 3 columns => independent)")
print(f"  Singular values: {np.linalg.svd(V_ind, compute_uv=False)}")
print(f"  All singular values bounded away from 0")

In [ ]:
# Visualize span and linear dependence in R^2 and R^3

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: R^2 - two independent vectors span the plane
ax = axes[0]
ax.set_title("Two independent vectors span $\\mathbb{R}^2$", fontsize=13)
ax.set_xlim(-1, 4); ax.set_ylim(-1, 3.5)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

u1, u2 = np.array([3, 1]), np.array([1, 2])
ax.annotate('', xy=u1, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='blue', lw=2.5))
ax.annotate('', xy=u2, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='red', lw=2.5))
ax.text(u1[0]+0.1, u1[1]+0.1, '$v_1$', color='blue', fontsize=14)
ax.text(u2[0]+0.1, u2[1]+0.1, '$v_2$', color='red', fontsize=14)

# Show the parallelogram they span
from matplotlib.patches import Polygon
parallelogram = Polygon([np.zeros(2), u1, u1+u2, u2], alpha=0.15, color='purple')
ax.add_patch(parallelogram)
ax.text(1.8, 1.2, 'span = $\\mathbb{R}^2$', fontsize=12, color='purple')

# Right: R^2 - two dependent vectors span only a line
ax = axes[1]
ax.set_title("Two dependent vectors span only a line", fontsize=13)
ax.set_xlim(-2, 4); ax.set_ylim(-1.5, 3)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

w1, w2 = np.array([2, 1]), np.array([3, 1.5])  # w2 = 1.5 * w1
ax.annotate('', xy=w1, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='blue', lw=2.5))
ax.annotate('', xy=w2, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='red', lw=2.5))
ax.text(w1[0]-0.3, w1[1]+0.2, '$v_1$', color='blue', fontsize=14)
ax.text(w2[0]+0.1, w2[1]+0.1, '$v_2 = 1.5 v_1$', color='red', fontsize=14)

t = np.linspace(-1, 2, 50)
ax.plot(t*2, t*1, 'purple', lw=1.5, alpha=0.5)
ax.text(0.5, -0.8, 'span = a line (1D)', fontsize=12, color='purple')

plt.tight_layout()
plt.show()

---
## 3. Linear Maps and Matrix Representations

### The key insight
A matrix $A \in \mathbb{R}^{m \times n}$ **is** a linear map $T: \mathbb{R}^n \to \mathbb{R}^m$, but its entries depend on your **choice of basis**. If $\{v_1, \dots, v_n\}$ is a basis for the domain and $\{w_1, \dots, w_m\}$ is a basis for the codomain, then $A_{ij}$ is the $i$-th coordinate of $T(v_j)$ in the $w$-basis.

### Change of basis
If $S$ is an invertible matrix whose columns are a new basis, then the matrix of the same linear map in the new basis is:

$$\tilde{A} = S^{-1} A S$$

This is a **similarity transformation**. Key fact: similar matrices have the same eigenvalues, determinant, trace, and rank.

### Why this matters for NLA
- **Diagonalization** $A = P \Lambda P^{-1}$ is a change of basis to the eigenvector basis where the map is just scaling.
- **The SVD** $A = U \Sigma V^T$ uses *two different* orthonormal bases (one for domain, one for codomain) to make the map as simple as possible.
- **Condition number** depends on the basis: the same linear map can be well-conditioned or catastrophically ill-conditioned depending on the coordinate system.

In [ ]:
# --- Linear Maps and Change of Basis ---
# A 2D rotation matrix in the standard basis
theta = np.pi / 4  # 45 degrees
A_rot = np.array([[np.cos(theta), -np.sin(theta)],
                  [np.sin(theta),  np.cos(theta)]])

print("Rotation matrix (standard basis):")
print(A_rot)
print(f"  Eigenvalues: {np.linalg.eigvals(A_rot)}")
print(f"  These are complex! Rotation has no real eigenvectors in R^2.\n")

# A shear matrix — same idea, very different geometry
A_shear = np.array([[1, 2],
                    [0, 1]])

print("Shear matrix:")
print(A_shear)
print(f"  Eigenvalues: {np.linalg.eigvals(A_shear)}")
print(f"  Both eigenvalues are 1, but the matrix is NOT the identity.")

In [ ]:
# Visualize how linear maps transform the unit circle
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

theta_circle = np.linspace(0, 2*np.pi, 200)
unit_circle = np.vstack([np.cos(theta_circle), np.sin(theta_circle)])

matrices = {
    'Rotation (45°)': A_rot,
    'Shear': A_shear,
    'Scaling': np.array([[2, 0], [0, 0.5]])
}

for ax, (name, M) in zip(axes, matrices.items()):
    transformed = M @ unit_circle
    
    ax.plot(unit_circle[0], unit_circle[1], 'b-', alpha=0.4, lw=1.5, label='Unit circle')
    ax.plot(transformed[0], transformed[1], 'r-', lw=2, label='Image')
    
    # Draw basis vector images
    e1, e2 = M @ np.array([1,0]), M @ np.array([0,1])
    ax.annotate('', xy=e1, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='darkred', lw=2))
    ax.annotate('', xy=e2, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='darkblue', lw=2))
    
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
    ax.set_title(name, fontsize=13)
    ax.legend(fontsize=9, loc='lower right')

plt.suptitle("Linear maps transform the unit circle into an ellipse (or degenerate case)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Change of Basis / Similarity Transformation ---
# A matrix in the standard basis
A = np.array([[5, 4],
              [2, 3]], dtype=float)

eigenvalues, P = np.linalg.eig(A)
print("A (standard basis):")
print(A)
print(f"\nEigenvalues: {eigenvalues}")
print(f"\nEigenvector matrix P (change-of-basis matrix):")
print(P)

# In the eigenvector basis, A becomes diagonal
Lambda = np.linalg.inv(P) @ A @ P
print(f"\nP^{{-1}} A P = Λ (diagonal in eigenbasis):")
print(Lambda)

# Verify similarity preserves trace and determinant
print(f"\nTrace preservation:  tr(A) = {np.trace(A):.4f},  tr(Λ) = {np.trace(Lambda):.4f}")
print(f"Det preservation:    det(A) = {np.linalg.det(A):.4f},  det(Λ) = {np.linalg.det(Lambda):.4f}")

---
## 4. Inner Products and Norms

This is where NLA diverges from pure linear algebra. Geometry enters through the inner product.

### Vector norms
A **norm** $\|\cdot\|: \mathbb{R}^n \to \mathbb{R}_{\ge 0}$ satisfies: (1) $\|x\| = 0 \iff x = 0$, (2) $\|\alpha x\| = |\alpha|\|x\|$, (3) $\|x+y\| \le \|x\| + \|y\|$.

| Norm | Formula | Use in NLA |
|---|---|---|
| $\ell_1$ | $\sum_i |x_i|$ | Sparsity, compressed sensing |
| $\ell_2$ (Euclidean) | $\sqrt{\sum_i x_i^2} = \sqrt{x^T x}$ | The default; tied to inner product and orthogonality |
| $\ell_\infty$ | $\max_i |x_i|$ | Componentwise error bounds |

### Inner product and orthogonality
The standard inner product $\langle x, y \rangle = x^T y$ induces the 2-norm: $\|x\|_2 = \sqrt{\langle x, x \rangle}$.

Two vectors are **orthogonal** if $x^T y = 0$. An **orthonormal** set satisfies $q_i^T q_j = \delta_{ij}$.

### Matrix norms
- **Frobenius norm:** $\|A\|_F = \sqrt{\sum_{i,j} a_{ij}^2} = \sqrt{\text{tr}(A^T A)}$. Treats the matrix as a long vector.
- **Induced (operator) 2-norm:** $\|A\|_2 = \max_{\|x\|_2=1} \|Ax\|_2 = \sigma_{\max}(A)$. The largest singular value.
- **Condition number:** $\kappa(A) = \|A\| \cdot \|A^{-1}\| = \sigma_{\max}/\sigma_{\min}$ (in the 2-norm).

Without these concepts, QR factorization, least squares, and convergence analysis are just symbol manipulation.

In [ ]:
# --- Vector Norms ---
x = np.array([3, -4, 0, 2, -1], dtype=float)

norm_1   = np.linalg.norm(x, 1)
norm_2   = np.linalg.norm(x, 2)
norm_inf = np.linalg.norm(x, np.inf)

print(f"x = {x}")
print(f"  ||x||_1   = {norm_1:.4f}   (sum of absolute values)")
print(f"  ||x||_2   = {norm_2:.4f}   (Euclidean length)")
print(f"  ||x||_inf = {norm_inf:.4f}   (max absolute value)")
print(f"\nNorm inequalities (always hold in R^n, n={len(x)}):")
print(f"  ||x||_inf <= ||x||_2 <= ||x||_1:  {norm_inf:.2f} <= {norm_2:.2f} <= {norm_1:.2f}  ✓")
print(f"  ||x||_1 <= sqrt(n)*||x||_2:        {norm_1:.2f} <= {np.sqrt(len(x))*norm_2:.2f}  ✓")
print(f"  ||x||_2 <= sqrt(n)*||x||_inf:      {norm_2:.2f} <= {np.sqrt(len(x))*norm_inf:.2f}  ✓")

In [ ]:
# Visualize unit balls in different norms (R^2)
fig, ax = plt.subplots(1, 1, figsize=(7, 7))

theta = np.linspace(0, 2*np.pi, 1000)

# l2 unit ball (circle)
ax.plot(np.cos(theta), np.sin(theta), 'b-', lw=2.5, label='$\\ell_2$ (circle)')

# l1 unit ball (diamond): |x| + |y| = 1
t = np.linspace(0, 1, 250)
l1_x = np.concatenate([t, -t, -t, t])
l1_y = np.concatenate([1-t, t-1, 1-t, t-1])
# Parametric: trace diamond
diamond_x = np.concatenate([t, -t[::-1], -t, t[::-1]])
diamond_y = np.concatenate([1-t, 1-t[::-1], -(1-t), -(1-t[::-1])])
ax.plot(diamond_x, diamond_y, 'r-', lw=2.5, label='$\\ell_1$ (diamond)')

# l_inf unit ball (square): max(|x|, |y|) = 1
square_x = [1, -1, -1, 1, 1]
square_y = [1, 1, -1, -1, 1]
ax.plot(square_x, square_y, 'g-', lw=2.5, label='$\\ell_\\infty$ (square)')

ax.set_xlim(-1.6, 1.6); ax.set_ylim(-1.6, 1.6)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
ax.set_title("Unit balls in $\\mathbb{R}^2$: $\\{x : \\|x\\|_p = 1\\}$", fontsize=14)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# --- Matrix Norms ---
M = np.array([[1, 2, 0],
              [0, 3, 1],
              [2, 0, 4]], dtype=float)

# Frobenius norm
frob = np.linalg.norm(M, 'fro')

# Induced 2-norm = largest singular value
sigma = np.linalg.svd(M, compute_uv=False)
norm2 = np.linalg.norm(M, 2)

# 1-norm and inf-norm (induced)
norm1 = np.linalg.norm(M, 1)       # max column sum
norminf = np.linalg.norm(M, np.inf) # max row sum

print(f"Matrix M:\n{M}\n")
print(f"Singular values:  {sigma}")
print(f"  ||M||_1     = {norm1:.4f}   (max column abs-sum)")
print(f"  ||M||_2     = {norm2:.4f}   (= σ_max = {sigma[0]:.4f})")
print(f"  ||M||_inf   = {norminf:.4f}   (max row abs-sum)")
print(f"  ||M||_F     = {frob:.4f}   (= sqrt(Σ σ_i²) = {np.sqrt(np.sum(sigma**2)):.4f})")
print(f"\nCondition number κ₂(M) = σ_max/σ_min = {sigma[0]/sigma[-1]:.4f}")

---
## 5. Orthogonal Projections

### Definition
An **orthogonal projector** onto a subspace $\mathcal{S}$ is a matrix $P$ satisfying:
1. **Idempotent:** $P^2 = P$ (projecting twice does nothing new)
2. **Symmetric:** $P^T = P$ (the projection is orthogonal, not oblique)

If $Q \in \mathbb{R}^{m \times k}$ has orthonormal columns spanning $\mathcal{S}$ (i.e., $Q^TQ = I_k$), then:

$$P = QQ^T$$

This projects any vector $b \in \mathbb{R}^m$ onto $\mathcal{S}$: the projection $Pb$ is the closest point in $\mathcal{S}$ to $b$, and the residual $(I - P)b$ is orthogonal to $\mathcal{S}$.

### Properties
- $\|Pb\|_2 \le \|b\|_2$ (projections don't increase length)
- $b - Pb \perp \mathcal{S}$ (the residual is orthogonal to the subspace)
- Eigenvalues of $P$ are only 0 and 1; $\text{rank}(P) = \text{tr}(P) = k$

### Why this matters for NLA
- **Least squares:** The solution minimizes $\|b - Ax\|_2$, which means $Ax^* = Pb$ where $P$ projects onto $\mathcal{C}(A)$.
- **Gram–Schmidt / QR:** Iteratively projects out components to build orthonormal bases.
- **Krylov methods:** Project the problem onto successively richer subspaces.

In [ ]:
# --- Orthogonal Projections ---
# Build an orthonormal basis Q for a 2D subspace of R^3
# then form P = Q Q^T

# Two vectors spanning a plane in R^3
a1 = np.array([1, 0, 1], dtype=float)
a2 = np.array([0, 1, 1], dtype=float)

# Orthonormalize via QR
Q, _ = np.linalg.qr(np.column_stack([a1, a2]))
print(f"Orthonormal basis Q (columns):\n{Q}\n")
print(f"Q^T Q = I check:\n{Q.T @ Q}\n")

# Projection matrix
P = Q @ Q.T
print(f"Projection matrix P = Q Q^T:\n{P}\n")

# Verify idempotency: P^2 = P
print(f"P^2 = P check (max diff): {np.max(np.abs(P @ P - P)):.2e}")

# Verify symmetry: P^T = P
print(f"P^T = P check (max diff): {np.max(np.abs(P.T - P)):.2e}")

# Eigenvalues of P should be 0 and 1
eigs_P = np.linalg.eigvalsh(P)
print(f"Eigenvalues of P: {eigs_P}")
print(f"rank(P) = tr(P) = {np.trace(P):.0f}")

In [ ]:
# Visualize orthogonal projection in R^3
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')

# The plane spanned by Q columns
s = np.linspace(-1.5, 1.5, 10)
S1, S2 = np.meshgrid(s, s)
plane_pts = np.array([Q[:, 0][i]*S1 + Q[:, 1][i]*S2 for i in range(3)])
ax.plot_surface(plane_pts[0], plane_pts[1], plane_pts[2], alpha=0.2, color='cyan')

# A vector b not in the plane
b = np.array([1, 2, 3], dtype=float)
Pb = P @ b          # projection
resid = b - Pb      # residual

# Plot b, Pb, and the residual
ax.quiver(0, 0, 0, *b, color='red', arrow_length_ratio=0.08, lw=2.5, label='$b$')
ax.quiver(0, 0, 0, *Pb, color='blue', arrow_length_ratio=0.08, lw=2.5, label='$Pb$ (projection)')
ax.quiver(*Pb, *resid, color='green', arrow_length_ratio=0.1, lw=2, linestyle='--', label='$b - Pb$ (residual)')

# Show right angle marker
ax.text(Pb[0]+0.1, Pb[1]+0.1, Pb[2]+0.3, '⊥', fontsize=16, color='green')

ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title('Orthogonal projection onto a 2D subspace of $\\mathbb{R}^3$', fontsize=13)
ax.legend(fontsize=11, loc='upper left')

print(f"b = {b}")
print(f"Pb = {Pb}")
print(f"Residual b - Pb = {resid}")
print(f"Residual ⊥ subspace check: Q^T(b-Pb) = {Q.T @ resid}")
print(f"||Pb||₂ = {np.linalg.norm(Pb):.4f} ≤ ||b||₂ = {np.linalg.norm(b):.4f}  ✓")

plt.tight_layout()
plt.show()

---
## 6. Eigenvalues, Eigenvectors, and the Spectral Theorem

### Definitions
A scalar $\lambda$ is an **eigenvalue** of $A \in \mathbb{R}^{n \times n}$ with **eigenvector** $v \ne 0$ if:

$$Av = \lambda v$$

The eigenvalues are roots of the **characteristic polynomial** $\det(A - \lambda I) = 0$.

### Key facts
- $\text{tr}(A) = \sum_i \lambda_i$ and $\det(A) = \prod_i \lambda_i$
- Similar matrices ($B = S^{-1}AS$) have the same eigenvalues
- A matrix is **diagonalizable** if it has $n$ linearly independent eigenvectors: $A = P\Lambda P^{-1}$

### The Spectral Theorem (the cornerstone)
If $A \in \mathbb{R}^{n \times n}$ is **symmetric** ($A = A^T$), then:
1. All eigenvalues are **real**
2. Eigenvectors corresponding to distinct eigenvalues are **orthogonal**
3. $A$ has a full set of orthonormal eigenvectors: $A = Q\Lambda Q^T$ where $Q$ is orthogonal ($Q^TQ = QQ^T = I$)

This is used *constantly* in NLA:
- Symmetric positive definite matrices arise in normal equations ($A^TA$), covariance matrices, and Hessians
- The SVD is essentially the spectral theorem applied to $A^TA$ and $AA^T$
- Convergence of iterative methods depends on the spectral radius $\rho(A) = \max|\lambda_i|$

In [ ]:
# --- Eigenvalues and Eigenvectors ---
# Non-symmetric matrix: eigenvalues can be complex, eigenvectors non-orthogonal
A_nonsym = np.array([[0, -1],
                     [1,  0]], dtype=float)  # 90° rotation

evals_ns, evecs_ns = np.linalg.eig(A_nonsym)
print("Non-symmetric matrix (rotation):")
print(A_nonsym)
print(f"  Eigenvalues: {evals_ns}  (complex!)")
print()

# Symmetric matrix: real eigenvalues, orthogonal eigenvectors
A_sym = np.array([[4, 1, 1],
                  [1, 3, 0],
                  [1, 0, 2]], dtype=float)

print(f"Symmetric? A = A^T: {np.allclose(A_sym, A_sym.T)}")
evals, evecs = np.linalg.eigh(A_sym)  # eigh for symmetric matrices
print(f"\nSymmetric matrix A:\n{A_sym}")
print(f"\nEigenvalues (all real): {evals}")
print(f"\nEigenvectors Q (columns):\n{evecs}")
print(f"\nOrthogonality check Q^T Q:\n{evecs.T @ evecs}")
print(f"\nSpectral decomposition check ||A - QΛQ^T||: {np.linalg.norm(A_sym - evecs @ np.diag(evals) @ evecs.T):.2e}")

In [ ]:
# Visualize eigenvectors of a symmetric matrix: A stretches along eigendirections
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

A2_sym = np.array([[3, 1],
                   [1, 2]], dtype=float)
evals2, evecs2 = np.linalg.eigh(A2_sym)

# Left: show how A maps the unit circle — eigenvectors align with ellipse axes
theta = np.linspace(0, 2*np.pi, 300)
circle = np.vstack([np.cos(theta), np.sin(theta)])
ellipse = A2_sym @ circle

ax = axes[0]
ax.plot(circle[0], circle[1], 'b-', alpha=0.4, lw=1.5, label='Unit circle')
ax.plot(ellipse[0], ellipse[1], 'r-', lw=2, label='$Ax$ (image)')

for i in range(2):
    ev = evecs2[:, i]
    lam = evals2[i]
    ax.annotate('', xy=lam*ev, xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color='darkgreen', lw=2.5))
    ax.annotate('', xy=ev, xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color='blue', lw=1.5))
    ax.text(lam*ev[0]+0.15, lam*ev[1]+0.15,
            f'$\\lambda_{i+1}v_{i+1}$ ($\\lambda={lam:.2f}$)',
            fontsize=11, color='darkgreen')

ax.set_xlim(-4.5, 4.5); ax.set_ylim(-4.5, 4.5)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.set_title('Symmetric matrix: eigenvectors are ellipse axes', fontsize=12)
ax.legend(fontsize=10)

# Right: spectral decomposition A = λ₁v₁v₁ᵀ + λ₂v₂v₂ᵀ
ax = axes[1]
rank1_components = []
for i in range(2):
    v = evecs2[:, i:i+1]
    component = evals2[i] * (v @ v.T)
    rank1_components.append(component)

labels = ['$A$', '=', f'$\\lambda_1 v_1 v_1^T$', '+', f'$\\lambda_2 v_2 v_2^T$']
matrices = [A2_sym, None, rank1_components[0], None, rank1_components[1]]

x_pos = 0
for label, mat in zip(labels, matrices):
    if mat is not None:
        ax.text(x_pos, 0.6, f'[{mat[0,0]:+.2f}  {mat[0,1]:+.2f}]\n[{mat[1,0]:+.2f}  {mat[1,1]:+.2f}]',
                fontsize=11, family='monospace', ha='center', va='center',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
        ax.text(x_pos, 1.15, label, fontsize=13, ha='center', va='center')
    else:
        ax.text(x_pos, 0.6, label, fontsize=18, ha='center', va='center')
    x_pos += 1.6

ax.set_xlim(-1, 8); ax.set_ylim(-0.2, 1.6)
ax.axis('off')
ax.set_title('Spectral decomposition: $A = \\sum \\lambda_i v_i v_i^T$', fontsize=12)

plt.tight_layout()
plt.show()

---
## 7. Rank and Near-Rank Deficiency

### Rank as a diagnostic
The **rank** of $A \in \mathbb{R}^{m \times n}$ is the dimension of $\mathcal{C}(A)$ (equivalently, of $\mathcal{C}(A^T)$). It equals the number of nonzero singular values.

$$\text{rank}(A) = \#\{\sigma_i > 0\}$$

### Rank deficiency
A matrix is **rank-deficient** if $\text{rank}(A) < \min(m, n)$. Consequences:
- $A\mathbf{x} = \mathbf{b}$ has either no solution or infinitely many solutions
- $A$ has a nontrivial null space
- $A^TA$ is singular (normal equations blow up)

### Near-rank deficiency: the numerical disaster
In floating-point arithmetic, exact rank deficiency is rare. Instead, we encounter matrices with **tiny singular values** — they are "nearly" rank-deficient. This is far more dangerous because:

$$\kappa_2(A) = \frac{\sigma_{\max}}{\sigma_{\min}} \gg 1$$

When $\kappa(A)$ is large:
- Small perturbations in $\mathbf{b}$ cause huge changes in the solution $\mathbf{x}$
- Gaussian elimination amplifies rounding errors
- The computed solution may have **no correct digits**

The SVD is the definitive tool for diagnosing and handling near-rank deficiency: truncate or regularize small singular values.

In [ ]:
# --- Rank and Rank Deficiency ---
# Exact rank deficiency
A_full = np.array([[1, 2], [3, 4]], dtype=float)       # rank 2
A_def  = np.array([[1, 2], [2, 4]], dtype=float)       # rank 1 (row 2 = 2*row 1)

print("Full-rank matrix:")
print(A_full)
print(f"  Rank = {np.linalg.matrix_rank(A_full)}, singular values = {np.linalg.svd(A_full, compute_uv=False)}")
print(f"  det = {np.linalg.det(A_full):.4f}, κ₂ = {np.linalg.cond(A_full):.4f}\n")

print("Rank-deficient matrix:")
print(A_def)
print(f"  Rank = {np.linalg.matrix_rank(A_def)}, singular values = {np.linalg.svd(A_def, compute_uv=False)}")
print(f"  det = {np.linalg.det(A_def):.4f}, κ₂ = {np.linalg.cond(A_def):.4e}")
print(f"  Null space: {null_space(A_def).flatten()}")

In [ ]:
# --- Near-Rank Deficiency: The Numerical Disaster ---
# The Hilbert matrix is the classic example of near-rank deficiency

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: singular value decay for Hilbert matrices of increasing size
ax = axes[0]
for n in [4, 6, 8, 10, 12]:
    H = np.array([[1.0/(i+j+1) for j in range(n)] for i in range(n)])
    sigmas = np.linalg.svd(H, compute_uv=False)
    ax.semilogy(range(1, n+1), sigmas, 'o-', label=f'n={n}', markersize=4)

ax.set_xlabel('Singular value index', fontsize=12)
ax.set_ylabel('$\\sigma_i$ (log scale)', fontsize=12)
ax.set_title('Singular value decay of Hilbert matrices', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Right: condition number growth
ax = axes[1]
sizes = range(2, 16)
conds = []
for n in sizes:
    H = np.array([[1.0/(i+j+1) for j in range(n)] for i in range(n)])
    conds.append(np.linalg.cond(H))

ax.semilogy(list(sizes), conds, 'ro-', markersize=6)
ax.set_xlabel('Matrix size $n$', fontsize=12)
ax.set_ylabel('$\\kappa_2(H_n)$ (log scale)', fontsize=12)
ax.set_title('Condition number of Hilbert matrices', fontsize=13)
ax.axhline(1/np.finfo(float).eps, color='gray', ls='--', label=f'$1/\\epsilon_{{mach}} \\approx {1/np.finfo(float).eps:.0e}$')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Machine epsilon: {np.finfo(float).eps:.2e}")
print(f"For n=12 Hilbert matrix: κ₂ ≈ {conds[10]:.2e}")
print(f"Expected digits of accuracy: {max(0, 16 - int(np.log10(conds[10])))}")

In [ ]:
# --- Sensitivity of linear systems to near-rank deficiency ---
# Solve Hx = b with a well-conditioned and ill-conditioned system

np.random.seed(42)

for n, label in [(5, "n=5 (moderate κ)"), (12, "n=12 (extreme κ)")]:
    H = np.array([[1.0/(i+j+1) for j in range(n)] for i in range(n)])
    x_true = np.ones(n)
    b = H @ x_true
    
    # Perturb b slightly (simulating measurement noise)
    delta_b = 1e-10 * np.random.randn(n)
    x_pert = np.linalg.solve(H, b + delta_b)
    
    rel_err_b = np.linalg.norm(delta_b) / np.linalg.norm(b)
    rel_err_x = np.linalg.norm(x_pert - x_true) / np.linalg.norm(x_true)
    kappa = np.linalg.cond(H)
    
    print(f"{label}:")
    print(f"  κ₂(H)           = {kappa:.2e}")
    print(f"  ||δb||/||b||     = {rel_err_b:.2e}  (tiny perturbation)")
    print(f"  ||δx||/||x_true||= {rel_err_x:.2e}  (amplified by κ!)")
    print(f"  Amplification    = {rel_err_x / rel_err_b:.2e}  (bounded by κ = {kappa:.2e})")
    print()

---
## Summary: How These Concepts Connect to NLA

| Prerequisite | Where it appears in NLA |
|---|---|
| **Four fundamental subspaces** | The SVD gives orthonormal bases for all four simultaneously |
| **Basis & dimension** | QR builds orthonormal bases; Krylov methods build them iteratively |
| **Linear maps & change of basis** | Diagonalization, SVD, and Schur decomposition are all change-of-basis operations |
| **Norms** | Error analysis, convergence criteria, and stopping conditions all require norms |
| **Orthogonal projections** | Least squares = projection onto $\mathcal{C}(A)$; Gram-Schmidt = successive projections |
| **Spectral theorem** | Foundation for symmetric eigenvalue algorithms; $A^TA$ and $AA^T$ are always symmetric |
| **Rank & conditioning** | Determines solvability, solution sensitivity, and which algorithms are safe to use |

**The unifying theme:** NLA algorithms succeed by finding *good bases* — orthonormal ones that reveal the geometric structure of linear maps while remaining numerically stable.

# TODO: Delete this cell (duplicate of cell 0)